# 3.5 总结：SASRec 与 Transformer 序列推荐

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

读取 SASRec 在真实 MovieLens 时间序列上的实验产物，与热门基线对照，并总结 SASRec、BERT4Rec、HSTU 的目标、信息边界和工程成本。

## Setup

默认 `smoke` 档使用仓库内固定版本的 GroupLens **MovieLens latest-small** 真实行为子集，CPU 可重复执行；`full` 档只扩大真实数据规模与训练配置，不切换到合成数据。数据包含真实匿名用户、电影、评分和时间戳；实验只做确定性截取与任务重构，不随机制造交互、标签或行为序列。原始许可与引用保存在 `data/ml-latest-small/README.txt`。

**主要资料：** [SASRec](https://arxiv.org/abs/1808.09781) · [BERT4Rec](https://arxiv.org/abs/1904.06690) · [HSTU](https://arxiv.org/abs/2402.17152)

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("RECSYS_PROFILE", "smoke")
PROFILE = os.environ["RECSYS_PROFILE"]
from recsys_lab.data import load_movielens, movielens_provenance
real_ratings, real_movies = load_movielens()
REAL_DATASET = movielens_provenance(real_ratings)
print({"profile": PROFILE, "root": str(ROOT), "real_dataset": REAL_DATASET})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'profile': 'smoke', 'root': '/workspace', 'real_dataset': {'dataset': 'MovieLens latest-small (GroupLens, generated 2018-09-26)', 'source': 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', 'license_file': '/workspace/data/ml-latest-small/README.txt', 'rows_used': 26732, 'users_used': 120, 'items_used': 600, 'time_min_utc': '1996-10-17T11:51:49+00:00', 'time_max_utc': '2018-09-13T21:38:16+00:00', 'positive_rule': 'like := observed rating >= 4.0; very_like := observed rating >= 4.5', 'randomly_fabricated_rows': 0}}


## Steps

## 1. 模型演进对比

| 模型 | Attention 可见范围 | 目标 | 推理输出 | 主要代价 |
|---|---|---|---|---|
| SASRec | 仅过去 | next-item pairwise | 最后位置向量 Top-K | $O(L^2)$ attention |
| BERT4Rec | 左右上下文 | masked item | mask 位置分布 | 训练—在线目标差异 |
| HSTU | 仅过去、推荐特化 | next-item | next-item / 行为流 | 模型—系统协同复杂 |

SASRec 是本章可执行主线；BERT4Rec 作为双向预训练分支介绍，HSTU 在 4.3 以生成式行为序列模型继续展开。

## 2. 真实实验结果

下表不手填数值，只读取 `3_5_1_sasrec` 在 MovieLens latest-small 真实时间序列上写出的 JSON。热门基线与 SASRec 使用相同测试目标。

In [2]:
import json, pandas as pd
result_path = ROOT / 'results' / 'chapter_3_5' / '3_5_1_sasrec.json'
payload = json.loads(result_path.read_text(encoding='utf-8'))
comparison = pd.DataFrame(payload['records'])
display(comparison.round(4))
assert len(comparison)==1
print('指标来源：', result_path.name)

,algorithm,primary_metric,primary_value,secondary_metric,secondary_value,baseline_metric,baseline_value,framework,source_notebook,dataset,randomly_fabricated_rows
0,SASRec 自注意力序列推荐,hr@10,0.0208,popularity_hr@10,0.0208,None,None,torch_rechub.models.matching.SASRec,3_5_1_sasrec,MovieLens latest-small,0


指标来源： 3_5_1_sasrec.json


## 3. 如何解释

- 若 SASRec 未超过热门基线，不应调换测试集或制造更容易的序列；应检查数据密度、负样本、训练轮数、序列长度和目标定义。
- MovieLens 是评分日志而不是连续曝光流，无法完整代表短视频、电商或广告的工业序列。
- 公平比较必须固定真实用户集合、时间切分、候选库、已见过滤和 K。
- 模型选型需同时记录 HR/NDCG、Coverage、P99、显存和增量更新成本。

## Checks

In [3]:
assert comparison.iloc[0]['dataset'] == 'MovieLens latest-small'
assert comparison.iloc[0]['randomly_fabricated_rows'] == 0
print('PASS：总结只聚合真实数据实验产物。')

PASS：总结只聚合真实数据实验产物。


## Next Steps

进入 4.0 比较 Transformer next-item 建模与 Semantic ID / 列表生成；在 4.3 使用相同真实时间序列观察 HSTU。